In [1]:
import pandas as pd
import os
from datetime import datetime
import re

C:\Users\sashe\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\sashe\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [9]:
# # import kagglehub
# Download latest version
# path = kagglehub.dataset_download("felipeesc/shark-attack-dataset")
# print("Path to dataset files:", path)

# files already downloaded so i saved it in a file
with open("path.txt", "r") as f:
    path = f.read()

In [4]:
os.listdir(path)

['attacks.csv', 'list_coor_australia.csv']

In [5]:
attacks_df = pd.read_csv(path+'/attacks.csv', encoding_errors='ignore')
list_coor_australia_df = pd.read_csv(path+'/list_coor_australia.csv')

In [6]:
len(attacks_df)

25723

In [7]:
list(attacks_df) # column titles

['Case Number',
 'Date',
 'Year',
 'Type',
 'Country',
 'Area',
 'Location',
 'Activity',
 'Name',
 'Sex ',
 'Age',
 'Injury',
 'Fatal (Y/N)',
 'Time',
 'Species ',
 'Investigator or Source',
 'pdf',
 'href formula',
 'href',
 'Case Number.1',
 'Case Number.2',
 'original order',
 'Unnamed: 22',
 'Unnamed: 23']

In [8]:
attacks_df.head()

,Case Number,Date,Year,Type,Country,Area,Location,Activity,Name,Sex,...,Species,Investigator or Source,pdf,href formula,href,Case Number.1,Case Number.2,original order,Unnamed: 22,Unnamed: 23
0,2018.06.25,25-Jun-2018,2018.0,Boating,USA,California,"Oceanside, San Diego County",Paddling,Julie Wolfe,F,...,White shark,"R. Collier, GSAF",2018.06.25-Wolfe.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2018.06.25,2018.06.25,6303.0,NaN,NaN
1,2018.06.18,18-Jun-2018,2018.0,Unprovoked,USA,Georgia,"St. Simon Island, Glynn County",Standing,AdysonMcNeely,F,...,NaN,"K.McMurray, TrackingSharks.com",2018.06.18-McNeely.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2018.06.18,2018.06.18,6302.0,NaN,NaN
2,2018.06.09,09-Jun-2018,2018.0,Invalid,USA,Hawaii,"Habush, Oahu",Surfing,John Denges,M,...,NaN,"K.McMurray, TrackingSharks.com",2018.06.09-Denges.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2018.06.09,2018.06.09,6301.0,NaN,NaN
3,2018.06.08,08-Jun-2018,2018.0,Unprovoked,AUSTRALIA,New South Wales,Arrawarra Headland,Surfing,male,M,...,2 m shark,"B. Myatt, GSAF",2018.06.08-Arrawarra.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2018.06.08,2018.06.08,6300.0,NaN,NaN
4,2018.06.04,04-Jun-2018,2018.0,Provoked,MEXICO,Colima,La Ticla,Free diving,Gustavo Ramos,M,...,"Tiger shark, 3m",A .Kipper,2018.06.04-Ramos.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2018.06.04,2018.06.04,6299.0,NaN,NaN


# STEPS: 

1. Eye ball data
- reveals that case number = date = case number 1 = case number 2, redundant with year
    - further analysis reveals taht case number is more standardised than date object so will keep it for now
- can delete case number 1 and 2 and year, convert date to datetime object. cases are uniquely identified by original order already
- there are trailing >20000 rows with mostly null or nan values except a 0 value in the first column
- what are the unnamed 1 and 2 columns? nothing important. can delete
- href, href formula, pdf columns are similar. href columns are "http://sharkattackfile.net/spreadsheets/pdf_directory/" + pdf column
- can delete href and href column

2. check if data types of columns are optimal or need to be converted to make sense. always remove leading and trailing spaces
3. check if data that doesn't make sense because it isn't standardised e.g. united states being represented as US in the same column


# Drop excessivly sparse rows or redundant columns

In [10]:
# drop duplicate or redundant columns. keep only one of the needed ones
attacks_df.drop(axis=1, labels=['Case Number.1','Case Number.2', 'Year'], inplace=True)

In [11]:
# NOTE these give high null percentages due to trailing 0s in first column
# print(attacks_df.isnull().all(axis=0)) # find columns with all nan --> there are none
print(attacks_df.isnull().sum()/len(attacks_df)) # count nan in columns and find fraction

Case Number               0.661704
Date                      0.755005
Type                      0.755161
Country                   0.756949
Area                      0.772694
Location                  0.775998
Activity                  0.776154
Name                      0.763169
Sex                       0.776970
Age                       0.865062
Injury                    0.756094
Fatal (Y/N)               0.775959
Time                      0.885394
Species                   0.865335
Investigator or Source    0.755666
pdf                       0.755005
href formula              0.755044
href                      0.755005
original order            0.754733
Unnamed: 22               0.999961
Unnamed: 23               0.999922
dtype: float64


In [12]:
# discover what unnamed columns represent by finding unique vaules
print(attacks_df['Unnamed: 22'].unique())
print(attacks_df['Unnamed: 23'].unique())

# drop sparse columns
attacks_df.drop(axis=1, labels=['Unnamed: 22', 'Unnamed: 23'], inplace=True)

<ArrowStringArray>
[nan, 'stopped here']
Length: 2, dtype: str
<ArrowStringArray>
[nan, 'Teramo', 'change filename']
Length: 3, dtype: str


In [13]:
# difference between href and href formula? 
(attacks_df['href'] == attacks_df['href formula']).sum() # 6242 of 6302 are the same. the fields approximate each other . though total number of rows is 22000, most are trailing 0s after aboht 6000
attacks_df.drop(axis=1, labels=['href', 'href formula'], inplace=True) # drop them, keep 'pdf'

In [14]:
# drop rows with mostly nan values
attacks_df.dropna(axis=0,thresh=7, inplace=True) # drop values with at least threshold 7 i.e. less than 40% of fields are non-na values

In [15]:
# check again
# print(attacks_df.isnull().all(axis=0)) # find columns with all nan --> there are none
print(attacks_df.isnull().sum()/len(attacks_df)) # count nan in columns and find fraction

Case Number               0.000159
Date                      0.000000
Type                      0.000635
Country                   0.007934
Area                      0.072199
Location                  0.085687
Activity                  0.086322
Name                      0.033323
Sex                       0.089654
Age                       0.449222
Injury                    0.004443
Fatal (Y/N)               0.085528
Time                      0.532212
Species                   0.450333
Investigator or Source    0.002698
pdf                       0.000000
original order            0.000000
dtype: float64


In [16]:
attacks_df

,Case Number,Date,Type,Country,Area,Location,Activity,Name,Sex,Age,Injury,Fatal (Y/N),Time,Species,Investigator or Source,pdf,original order
0,2018.06.25,25-Jun-2018,Boating,USA,California,"Oceanside, San Diego County",Paddling,Julie Wolfe,F,57,"No injury to occupant, outrigger canoe and pad...",N,18h00,White shark,"R. Collier, GSAF",2018.06.25-Wolfe.pdf,6303.0
1,2018.06.18,18-Jun-2018,Unprovoked,USA,Georgia,"St. Simon Island, Glynn County",Standing,AdysonMcNeely,F,11,Minor injury to left thigh,N,14h00 -15h00,NaN,"K.McMurray, TrackingSharks.com",2018.06.18-McNeely.pdf,6302.0
2,2018.06.09,09-Jun-2018,Invalid,USA,Hawaii,"Habush, Oahu",Surfing,John Denges,M,48,Injury to left lower leg from surfboard skeg,N,07h45,NaN,"K.McMurray, TrackingSharks.com",2018.06.09-Denges.pdf,6301.0
3,2018.06.08,08-Jun-2018,Unprovoked,AUSTRALIA,New South Wales,Arrawarra Headland,Surfing,male,M,NaN,Minor injury to lower leg,N,NaN,2 m shark,"B. Myatt, GSAF",2018.06.08-Arrawarra.pdf,6300.0
4,2018.06.04,04-Jun-2018,Provoked,MEXICO,Colima,La Ticla,Free diving,Gustavo Ramos,M,NaN,Lacerations to leg & hand shark PROVOKED INCIDENT,N,NaN,"Tiger shark, 3m",A .Kipper,2018.06.04-Ramos.pdf,6299.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6297,ND.0005,Before 1903,Unprovoked,AUSTRALIA,Western Australia,Roebuck Bay,Diving,male,M,NaN,FATAL,Y,NaN,NaN,"H. Taunton; N. Bartlett, p. 234",ND-0005-RoebuckBay.pdf,6.0
6298,ND.0004,Before 1903,Unprovoked,AUSTRALIA,Western Australia,NaN,Pearl diving,Ahmun,M,NaN,FATAL,Y,NaN,NaN,"H. Taunton; N. Bartlett, pp. 233-234",ND-0004-Ahmun.pdf,5.0
6299,ND.0003,1900-1905,Unprovoked,USA,North Carolina,Ocracoke Inlet,Swimming,Coast Guard personnel,M,NaN,FATAL,Y,NaN,NaN,"F. Schwartz, p.23; C. Creswell, GSAF",ND-0003-Ocracoke_1900-1905.pdf,4.0
6300,ND.0002,1883-1889,Unprovoked,PANAMA,NaN,"Panama Bay 8N, 79W",NaN,Jules Patterson,M,NaN,FATAL,Y,NaN,NaN,"The Sun, 10/20/1938",ND-0002-JulesPatterson.pdf,3.0


# Convert data to accurate data types, remove leading or trailing spaces

In [17]:
attacks_df.dtypes # find data types

Case Number                   str
Date                          str
Type                          str
Country                       str
Area                          str
Location                      str
Activity                      str
Name                          str
Sex                           str
Age                           str
Injury                        str
Fatal (Y/N)                   str
Time                          str
Species                       str
Investigator or Source        str
pdf                           str
original order            float64
dtype: object

In [ ]:
# remove leading and trailing spaces

## convert str to date type for 'Date' value

# Notes
-> In 'Date' column:

most are dd-mmm-yyyy

some are "Reported dd-mmm-yyyy" month in text

some are "yyyy.dd.mm" month in numbers

some are 'Fall 2008' or 'Summer-2008'

some are 'Early Aug-2006'

19-Jul-2004 Reported to have happened  "on the weekend


-> seems like 'Case Number' column is way more standardized than 'Date' column

yyyy.mm.dd(.x)(.R) 
- where mm can be 00 if the month isn't known and dd can also by 00 if the day isn't known
- where .R apprears if the instance was reported on the given day but did not necessarily happen that day
- where .x can be any lower case letter to differentiate between two or more instances that happened on the same day e.g. 2018.05.13.b and 2018.05.13.a

0000.yyyy where year is B.C.

ND-nnnn or ND.nnnn
- where nnnn is a four digit number that uniquely identifies an undated incident e.g. 'Between   1951-1963', '2009?'

In [27]:
'''
To find how many months or days are unknown out of 6000 -> to see if they are so few they can be discarded

This cell is mostly LLM generated code which I modified
'''

missing_month_pattern = r'^\d{4}\.00\.\d{2}(?:\.[a-z])?(?:\.R)?$'
missing_day_pattern = r'^\d{4}\.\d{2}\.00(?:\.[a-z])?(?:\.R)?$'

def count_date_matching_pattern(df, pattern, case_column='Case Number'):
    """
    Count the number of case numbers with unknown months (mm = 00)
    in the format yyyy.mm.dd(.x)(.R)
    
    Parameters:
    df (pd.DataFrame): Input dataframe
    case_column (str): Name of the column containing case numbers
    
    Returns:
    int: Count of unknown months
    """
    count = 0
    matching_cases = []
    
    for case in df[case_column].dropna():
        case_str = str(case).strip()
        
        # Pattern for yyyy.00.dd format with optional suffixes
        # This looks for month = 00 in the date pattern
        
        
        if re.match(pattern, case_str):
            count += 1
            matching_cases.append(case_str)
    
    return count, matching_cases

In [26]:
print("Unknown days:", count_date_matching_pattern(attacks_df, missing_day_pattern)[0])
print("Unknown months:", count_date_matching_pattern(attacks_df, missing_month_pattern)[0])

Unknown days: 742
Unknown months: 397


In [28]:
'''
This cell is LLM generated but modified by me. 

prompt for LLM to write initial code:
Hey, I am working on a dataset in Python pandas. I am trying to create a datetime column named "Date (datetime)" from an existing string "Case Number" column. 
The format of the data in "Case Number" is either:
yyyy.mm.dd(.x)(.R) 
- where mm can be 00 if the month isn't known and dd can also by 00 if the day isn't known
- where .R apprears if the instance was reported on the given day but did not necessarily happen that day
- where .x can be any lower case letter to differentiate between two or more instances that happened on the same day e.g. 2018.05.13.b and 2018.05.13.a

0000.yyyy where year is B.C.

ND-nnnn or ND.nnnn
- where nnnn is a four digit number that uniquely identifies an undated incident e.g. 'Between   1951-1963', '2009?'

Could you write me code or a function to create a new datetime column named "Date (datetime)"?
'''
def parse_case_number_to_datetime(case_number):
    """
    Convert case number string to datetime object based on various formats.
    
    Parameters:
    case_number (str): The case number string to parse
    
    Returns:
    datetime or NaT: Parsed datetime or NaT if parsing fails
    """
    if pd.isna(case_number) or case_number is None:
        return pd.NaT
    
    # Convert to string and strip whitespace
    case_str = str(case_number).strip()
    
    # Handle empty strings
    if case_str == '':
        return pd.NaT
    
    # Pattern 1: yyyy.mm.dd format with optional .x and .R suffixes
    # Examples: 2018.05.13, 2018.05.13.b, 2018.05.13.R, 2018.05.13.b.R
    pattern1 = r'^(\d{4})\.(\d{2})\.(\d{2})(?:\.([a-z]))?(?:\.(R))?$'
    match = re.match(pattern1, case_str)
    
    if match:
        year, month, day, letter, reported = match.groups()
        year = int(year)
        month = int(month)
        day = int(day)
        
        # Handle unknown month or day (00)
        if month == 0:
            return pd.NaT # unknown month is only 300/6302, <5% so discard
        if day == 0:
            day = 1    # keeping known months but unknown day seems more important since 742/6302, >10% are affected. this won't negatively affect analysis anyway
            
        try:
            return datetime(year, month, day)
        except ValueError:
            return pd.NaT
    
    # Pattern 2: 0000.yyyy (BC years)
    pattern2 = r'^0000\.(\d{4})$'
    match = re.match(pattern2, case_str)
    
    if match:
        year = int(match.group(1))
        return pd.NaT  # BC dates can't be directly represented in pandas datetime
    
    # Pattern 3: ND-nnnn or ND.nnnn (Undated incidents)
    pattern3 = r'^ND[.-](\d{4})$'
    match = re.match(pattern3, case_str)
    
    if match:
        return pd.NaT

    # If no patterns match, return NaT
    return pd.NaT

In [29]:
attacks_df['Date (datetime)'] = attacks_df['Case Number'].apply(parse_case_number_to_datetime)

In [30]:
attacks_df

,Case Number,Date,Type,Country,Area,Location,Activity,Name,Sex,Age,Injury,Fatal (Y/N),Time,Species,Investigator or Source,pdf,original order,Date (datetime)
0,2018.06.25,25-Jun-2018,Boating,USA,California,"Oceanside, San Diego County",Paddling,Julie Wolfe,F,57,"No injury to occupant, outrigger canoe and pad...",N,18h00,White shark,"R. Collier, GSAF",2018.06.25-Wolfe.pdf,6303.0,2018-06-25
1,2018.06.18,18-Jun-2018,Unprovoked,USA,Georgia,"St. Simon Island, Glynn County",Standing,AdysonMcNeely,F,11,Minor injury to left thigh,N,14h00 -15h00,NaN,"K.McMurray, TrackingSharks.com",2018.06.18-McNeely.pdf,6302.0,2018-06-18
2,2018.06.09,09-Jun-2018,Invalid,USA,Hawaii,"Habush, Oahu",Surfing,John Denges,M,48,Injury to left lower leg from surfboard skeg,N,07h45,NaN,"K.McMurray, TrackingSharks.com",2018.06.09-Denges.pdf,6301.0,2018-06-09
3,2018.06.08,08-Jun-2018,Unprovoked,AUSTRALIA,New South Wales,Arrawarra Headland,Surfing,male,M,NaN,Minor injury to lower leg,N,NaN,2 m shark,"B. Myatt, GSAF",2018.06.08-Arrawarra.pdf,6300.0,2018-06-08
4,2018.06.04,04-Jun-2018,Provoked,MEXICO,Colima,La Ticla,Free diving,Gustavo Ramos,M,NaN,Lacerations to leg & hand shark PROVOKED INCIDENT,N,NaN,"Tiger shark, 3m",A .Kipper,2018.06.04-Ramos.pdf,6299.0,2018-06-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6297,ND.0005,Before 1903,Unprovoked,AUSTRALIA,Western Australia,Roebuck Bay,Diving,male,M,NaN,FATAL,Y,NaN,NaN,"H. Taunton; N. Bartlett, p. 234",ND-0005-RoebuckBay.pdf,6.0,NaT
6298,ND.0004,Before 1903,Unprovoked,AUSTRALIA,Western Australia,NaN,Pearl diving,Ahmun,M,NaN,FATAL,Y,NaN,NaN,"H. Taunton; N. Bartlett, pp. 233-234",ND-0004-Ahmun.pdf,5.0,NaT
6299,ND.0003,1900-1905,Unprovoked,USA,North Carolina,Ocracoke Inlet,Swimming,Coast Guard personnel,M,NaN,FATAL,Y,NaN,NaN,"F. Schwartz, p.23; C. Creswell, GSAF",ND-0003-Ocracoke_1900-1905.pdf,4.0,NaT
6300,ND.0002,1883-1889,Unprovoked,PANAMA,NaN,"Panama Bay 8N, 79W",NaN,Jules Patterson,M,NaN,FATAL,Y,NaN,NaN,"The Sun, 10/20/1938",ND-0002-JulesPatterson.pdf,3.0,NaT


In [35]:
# find number of dates in column of 6302
print(len(attacks_df[attacks_df['Date (datetime)'].notna()]), "non-NaT dates of", len(attacks_df), "total")

5733 non-NaT dates of 6302 total
